# Rimless wheel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bernardorivas/hybrid_boxmap/blob/main/notebooks/rimless_wheel.ipynb)

The system is given by

$$\ddot\theta=\sin\theta,$$

with guard and reset

$$G=\{(\alpha+\gamma,\dot\theta): \dot\theta\geq 0\},\qquad r(\theta,\dot\theta)=(2\gamma-\theta,\cos(2\alpha)\,\dot\theta),$$

where $\gamma=0.2$ and $2\alpha=0.8$.

Stored figures are redrawn below. [Recompute from scratch](rimless_wheel_from_scratch.ipynb).


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or newer is required; select a current Colab runtime.")

if "google.colab" in sys.modules:
    repo = Path("/content/hybrid_boxmap")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bernardorivas/hybrid_boxmap.git", str(repo)], check=True)
    # Installation also runs when the checkout already exists.
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "-e", str(repo),
        "cmgdb==1.3.3+fork.6", "--find-links",
        "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.6",
    ], check=True)
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "code"]
    repo = next((p for p in candidates if (p / "demo/run_paper_examples.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Open this notebook from the hybrid_boxmap checkout.")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if shutil.which("dot") is None:
    raise RuntimeError("Install Graphviz (the dot executable) before plotting Morse graphs.")
import CMGDB
from hybrid_dynamics.src.carrier_kernel import cmgdb_provenance
if not hasattr(CMGDB, "ComputeCarrierChainMap"):
    raise RuntimeError("Install CMGDB 1.3.3+fork.6 as described in the README.")
print("Repository:", repo)
print("CMGDB:", cmgdb_provenance("native"))


## Phase portrait

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hybrid_dynamics.examples.paper_examples import paper_problem

problem = paper_problem("rimless-wheel")

fig, ax = plt.subplots(figsize=(5, 4))
for state in [(0.0, 0.05), (0.0, 0.2), (0.0, 0.9), (-0.2, 0.25), (0.3, -0.3), (0.3, -0.15), (-0.2, 1.0)]:
    trajectory = problem.system.simulate(np.array(state, float), (0.0, 10.0), max_jumps=30, max_step=0.01)
    color = None
    for segment in trajectory.segments:
        line, = ax.plot(*np.asarray(segment.state_values).T, lw=0.8, color=color)
        color = line.get_color()
ax.plot([0.6, 0.6], [0, 1], color="k", lw=2.5)  # guard
ax.set(xlim=(-0.2, 0.62), ylim=(-0.5, 1.0), xlabel="$\\theta$", ylabel="$\\dot\\theta$")
plt.show()

## Morse sets

Stored computation: $\tau=0.5$, level $n=7$, offset $K=0$, $128^2$ base cells, and 512 phase cells on $R=[-0.2,0.6]\times[-0.5,1]$.

Base cells per axis are $2^{n+K}$; phase cells are $2^{n+2}$. Gap refinement depth is 14.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import Image, display

output_dir = Path("output") / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
output_dir.mkdir(parents=True, exist_ok=False)
print("Output:", output_dir.resolve())


The coarse and fine panels use the same axes and limits. The fine record has $2048^2$ base cells and the same 512 phase cells. Their full Morse graphs have 20 and 17 nodes; both restricted graphs consist of the saddle $(0,1,0)$ above the orbit $(1,1,0)$.

In [ ]:
from demo.paper_comparison import render_wheel_comparison

record = Path("figures/paper/paper-rimless-wheel-tau050-level7-corners-gap-refined.json")
fine_record = Path("figures/paper/paper-rimless-wheel-tau050-level7-base2048-corners-gap-refined.json")
figures = render_wheel_comparison(record, fine_record, output_dir, titles=False)
display(Image(figures["comparison_files"][1], width=1100))


## Full Conley–Morse graphs

Morse sets with trivial index are gray.

In [ ]:
display(Image(figures["full_graph_comparison_files"][1], width=900))


In [ ]:
import json
from hybrid_dynamics.examples.paper_figures import conley_index_dimensions

def report_record(record, wall_seconds=None):
    summary = json.loads(Path(record).read_text())
    variant = summary["figure_variants"]["nontrivial"]
    print(f"Base: {summary['base_cells_per_axis']}²; phase cells: {summary['phase_cells']}; "
          f"Morse sets: {len(summary['morse_graph']['nodes'])}")
    print("Restricted order:", variant["order_shown"])
    if wall_seconds is not None:
        print(f"Wall time: {wall_seconds:.2f} s")
    print("Node | index (degrees 0, 1, 2) | construction")
    for entry in summary["conley"]:
        if entry["computed"]:
            dimensions = conley_index_dimensions(entry["shift_class"])
            label = dimensions[:3] if dimensions is not None else entry["shift_class"]
        else:
            label = "BLOCKED: " + entry.get("blocker", "see record")
        print(entry["morse_node"], "|", label, "|", entry.get("label_source", ""))
        if entry.get("exit_components_blocker"):
            print("  auto fallback:", entry["exit_components_blocker"])
    print("Record:", Path(record).resolve())
    return summary

summary = report_record(record)
fine_summary = report_record(fine_record)
